# Notebook 06 v3 — Cross-Model SHAP Agreement on Canonical Shared Samples

**Purpose:** Apply the six Krishna et al. (2022, TMLR) cross-model agreement metrics to SHAP outputs from Notebook 04c (canonical shared sample selection). Test whether different model architectures (RF, XGB, DNN) agree on feature importance for the same inputs.

**Why v3 not v2:** The earlier v2 attempt revealed that Notebook 04's SHAP outputs used per-model random sample selections — different architectures evaluated different 1000-sample subsets. Cross-architecture overlap was small (~100-235 samples per pair, dominated by rare classes), making Krishna metrics meaningless. Notebook 04c fixed this by recomputing SHAP on canonical per-dataset shared sample indices. This v3 of Notebook 06 reads those `_shap_shared.npy` outputs.

**Per-class sample counts (canonical, identical across architectures within each dataset):**
- NSL: Normal=261, DoS=248, Probe=211, R2L=213, U2R=67
- UNSW: 200 per class (all balanced)
- CIC: Normal=354, DoS=230, Probe=209, R2L=200, U2R=7

**Scope:**
- 3 datasets × 6 model variants × 3 architecture pairs × 3 K values × (aggregate + 5 per-class)
- Per-sample disagreement scores for Day 5 LLM evaluation pipeline

**Six Krishna metrics (verbatim from `06_shap_agreement.ipynb`):** FA, RA, SA, SRA, RC, PRA

**Predictions to test (from v7 §12.7):**
1. Tree-tree agreement (RF-XGB) > tree-DNN agreement (RF-DNN, XGB-DNN)
2. Agreement drops on rare classes (R2L, U2R) vs common (Normal, DoS)

**Prerequisite:** Notebook 04c must have completed successfully. Verify by checking `shap_values/{ds}/{model}_shap_shared.npy` files exist for all 18 models.

**Time estimate:** ~5-10 min compute (pure numpy + scipy, no GPU).

**Outputs:**
- `results/tables/krishna_agreement_canonical.csv` (model-pair-level)
- `results/tables/sample_disagreement_canonical.csv` (per-sample, for Day 5)
- `results/tables/krishna_agreement_canonical_summary.json` (aggregate + prediction tests)

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
MODEL_VARIANTS = ['5class_cw', '5class_smote']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
K_VALUES = [5, 10, 15]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
PAIRS = [('rf', 'xgb'), ('rf', 'dnn'), ('xgb', 'dnn')]

print(f'Scope: {len(DATASETS)} datasets × {len(MODEL_VARIANTS)} variants × {len(PAIRS)} pairs × {len(K_VALUES)} K values')
print(f'      = {len(DATASETS) * len(MODEL_VARIANTS) * len(PAIRS) * len(K_VALUES)} pair-K cells (aggregate)')
print(f'      = {len(DATASETS) * len(MODEL_VARIANTS) * len(PAIRS) * len(K_VALUES) * 5} pair-K-class cells (per-class)')

Scope: 3 datasets × 2 variants × 3 pairs × 3 K values
      = 54 pair-K cells (aggregate)
      = 270 pair-K-class cells (per-class)


## 2. Six Krishna metrics (verbatim from `06_shap_agreement.ipynb`)

In [3]:
def feature_agreement(shap_a, shap_b, k):
    """|top_k(A) ∩ top_k(B)| / k. Range [0, 1]."""
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    return len(top_a & top_b) / k

def rank_agreement(shap_a, shap_b, k):
    """Fraction of top-k features in same rank position in both."""
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    same_rank = sum(1 for i in range(k) if rank_a[i] == rank_b[i])
    return same_rank / k

def sign_agreement(shap_a, shap_b, k):
    """Of top-k features common to both, what fraction have same sign?"""
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    common = top_a & top_b
    if not common:
        return 0.0
    same_sign = sum(1 for f in common if np.sign(shap_a[f]) == np.sign(shap_b[f]))
    return same_sign / len(common)

def signed_rank_agreement(shap_a, shap_b, k):
    """Fraction of top-k positions where same feature AND same sign."""
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    matches = sum(1 for i in range(k)
                  if rank_a[i] == rank_b[i] and np.sign(shap_a[rank_a[i]]) == np.sign(shap_b[rank_a[i]]))
    return matches / k

def rank_correlation(shap_a, shap_b, k):
    """Spearman correlation between rankings of top-k features in A."""
    top_a_idx = np.argsort(-np.abs(shap_a))[:k]
    ranks_a = np.argsort(np.argsort(-np.abs(shap_a)[top_a_idx])) + 1
    ranks_b = np.argsort(np.argsort(-np.abs(shap_b)[top_a_idx])) + 1
    if k < 2:
        return 0.0
    corr, _ = spearmanr(ranks_a, ranks_b)
    return float(corr) if not np.isnan(corr) else 0.0

def pairwise_rank_agreement(shap_a, shap_b, k):
    """For each pair in top-k(A), do both explainers agree on which is more important?"""
    top_a = np.argsort(-np.abs(shap_a))[:k]
    abs_a = np.abs(shap_a)
    abs_b = np.abs(shap_b)
    n_pairs = 0
    agree = 0
    for i in range(k):
        for j in range(i+1, k):
            fi, fj = top_a[i], top_a[j]
            n_pairs += 1
            if abs_b[fi] >= abs_b[fj]:
                agree += 1
    return agree / n_pairs if n_pairs > 0 else 0.0

METRICS = {
    'FA': feature_agreement,
    'RA': rank_agreement,
    'SA': sign_agreement,
    'SRA': signed_rank_agreement,
    'RC': rank_correlation,
    'PRA': pairwise_rank_agreement,
}
print(f'✓ Six Krishna metrics defined: {list(METRICS.keys())}')

✓ Six Krishna metrics defined: ['FA', 'RA', 'SA', 'SRA', 'RC', 'PRA']


## 3. SHAP array helpers

In [4]:
def get_5class_aggregate_vector(shap_arr, sample_i):
    """Sum |SHAP| across classes for aggregate importance. Returns (n_features,)."""
    return np.abs(shap_arr[sample_i]).sum(axis=-1)

def get_5class_per_class_vector(shap_arr, sample_i, c):
    """SHAP values for class c. Returns (n_features,)."""
    return shap_arr[sample_i, :, c]

def compute_pair_metrics(arr_a, arr_b, k, sample_indices, mode, class_c=None):
    """Compute all six metrics for one model pair, averaged across samples.
    Also returns per-sample metric dicts.

    Args:
        arr_a, arr_b: (n_samples, n_features, n_classes) SHAP arrays — MUST be on same samples
        k: top-k value
        sample_indices: positions within the SHAP arrays to evaluate
        mode: 'aggregate' or 'perclass'
        class_c: class index for perclass mode
    """
    if len(sample_indices) == 0:
        return {m: np.nan for m in METRICS}, []

    out = {m: 0.0 for m in METRICS}
    per_sample = []

    for i in sample_indices:
        if mode == 'aggregate':
            sv_a = get_5class_aggregate_vector(arr_a, i)
            sv_b = get_5class_aggregate_vector(arr_b, i)
        else:
            sv_a = get_5class_per_class_vector(arr_a, i, class_c)
            sv_b = get_5class_per_class_vector(arr_b, i, class_c)

        sample_metrics = {}
        for m, fn in METRICS.items():
            v = fn(sv_a, sv_b, k)
            out[m] += v
            sample_metrics[m] = v
        per_sample.append((i, sample_metrics))

    avg = {m: out[m] / len(sample_indices) for m in METRICS}
    return avg, per_sample

print('✓ SHAP helpers loaded')

✓ SHAP helpers loaded


## 4. Load CANONICAL SHAP arrays + verify alignment

In [5]:
shap_arrays = {}  # {(dataset, variant): {arch: array}}
y_test_at_eval = {}  # {dataset: array of class labels at canonical eval indices}

all_aligned = True

for ds in DATASETS:
    # Load canonical eval indices for this dataset
    canonical_path = Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy'
    if not canonical_path.exists():
        print(f'❌ {ds}: canonical_eval_idx.npy missing — Notebook 04c not complete?')
        all_aligned = False
        continue

    canonical_eval_idx = np.load(canonical_path)
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    y_test_at_eval[ds] = y_test[canonical_eval_idx]

    for variant in MODEL_VARIANTS:
        key = (ds, variant)
        shap_arrays[key] = {}

        for arch in ARCHITECTURES:
            model_name = f'{arch}_{variant}'
            shap_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_shap_shared.npy'
            idx_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_eval_idx_shared.npy'

            if not shap_path.exists():
                print(f'  ❌ Missing: {shap_path.relative_to(REPO)}')
                all_aligned = False
                continue

            # Verify this model uses canonical indices
            saved_idx = np.load(idx_path)
            if not np.array_equal(saved_idx, canonical_eval_idx):
                print(f'  ❌ {model_name}: eval_idx_shared does NOT match canonical')
                all_aligned = False
                continue

            shap_arrays[key][arch] = np.load(shap_path)

        if len(shap_arrays[key]) == 3:
            shapes = [shap_arrays[key][a].shape for a in ARCHITECTURES]
            print(f'  ✓ {ds:<18} {variant:<14} n={shapes[0][0]}, shape={shapes[0]}, all 3 archs aligned on canonical indices')
        else:
            print(f'  ⚠ {ds:<18} {variant:<14} only {len(shap_arrays[key])}/3 architectures loaded')
            all_aligned = False

print(f'\n{"="*70}')
print(f'ALIGNMENT VERIFICATION: {"✓ ALL 18 MODELS ALIGNED" if all_aligned else "❌ MISALIGNMENT DETECTED"}')
print(f'{"="*70}')

assert all_aligned, 'Alignment check failed — do not proceed until 04c completes successfully for all 18 models'

  ✓ nsl_kdd_v2         5class_cw      n=1000, shape=(1000, 122, 5), all 3 archs aligned on canonical indices
  ✓ nsl_kdd_v2         5class_smote   n=1000, shape=(1000, 122, 5), all 3 archs aligned on canonical indices
  ✓ unsw_nb15_v2       5class_cw      n=1000, shape=(1000, 194, 5), all 3 archs aligned on canonical indices
  ✓ unsw_nb15_v2       5class_smote   n=1000, shape=(1000, 194, 5), all 3 archs aligned on canonical indices
  ✓ cic_ids2017_v2     5class_cw      n=1000, shape=(1000, 78, 5), all 3 archs aligned on canonical indices
  ✓ cic_ids2017_v2     5class_smote   n=1000, shape=(1000, 78, 5), all 3 archs aligned on canonical indices

ALIGNMENT VERIFICATION: ✓ ALL 18 MODELS ALIGNED


## 5. Main loop: compute all pair × K × (aggregate + per-class) metrics

In [6]:
agg_results = []
perclass_results = []
sample_disagreement_records = []  # per-sample disagreement for K=10 aggregate

t_overall = time.time()
print(f'\n{"="*70}')
print(f'Cross-model agreement computation (canonical samples) — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    for variant in MODEL_VARIANTS:
        key = (ds, variant)
        if key not in shap_arrays or len(shap_arrays[key]) < 3:
            continue

        print(f'\n=== {ds} / {variant} ===')
        y_eval = y_test_at_eval[ds]
        all_sample_idx = np.arange(shap_arrays[key]['rf'].shape[0])

        for (arch_a, arch_b) in PAIRS:
            arr_a = shap_arrays[key][arch_a]
            arr_b = shap_arrays[key][arch_b]

            for k in K_VALUES:
                # Aggregate mode (all 1000 samples)
                avg, per_sample = compute_pair_metrics(
                    arr_a, arr_b, k, all_sample_idx, 'aggregate'
                )
                row = {
                    'dataset': ds, 'variant': variant,
                    'pair': f'{arch_a}-{arch_b}', 'k': k, 'class': 'aggregate',
                }
                row.update(avg)
                agg_results.append(row)

                # For K=10, save per-sample disagreement scores
                if k == 10:
                    for sample_pos, sample_metrics in per_sample:
                        mean_agreement = np.mean(list(sample_metrics.values()))
                        sample_disagreement_records.append({
                            'dataset': ds, 'variant': variant,
                            'pair': f'{arch_a}-{arch_b}',
                            'sample_position': sample_pos,
                            'true_class': int(y_eval[sample_pos]),
                            'mean_agreement': float(mean_agreement),
                            'disagreement_score': float(1.0 - mean_agreement),
                            **{f'sample_{m}': float(v) for m, v in sample_metrics.items()},
                        })

                # Per-class mode (5 classes)
                for c in range(5):
                    class_idx = np.where(y_eval == c)[0]
                    if len(class_idx) < 5:
                        continue
                    avg_c, _ = compute_pair_metrics(
                        arr_a, arr_b, k, class_idx, 'perclass', class_c=c
                    )
                    row_c = {
                        'dataset': ds, 'variant': variant,
                        'pair': f'{arch_a}-{arch_b}', 'k': k,
                        'class': CLASS_NAMES_5[c], 'class_idx': c,
                        'n_samples': len(class_idx),
                    }
                    row_c.update(avg_c)
                    perclass_results.append(row_c)

            k10_row = [r for r in agg_results if r['dataset']==ds and r['variant']==variant
                       and r['pair']==f'{arch_a}-{arch_b}' and r['k']==10][0]
            print(f'  {arch_a}-{arch_b:<5} K=10 agg: ' +
                  ', '.join(f'{m}={k10_row[m]:.3f}' for m in ['FA', 'RA', 'RC', 'PRA']))

elapsed = (time.time() - t_overall) / 60
print(f'\nTotal time: {elapsed:.1f} min')
print(f'Aggregate rows: {len(agg_results)}')
print(f'Per-class rows: {len(perclass_results)}')
print(f'Sample disagreement records: {len(sample_disagreement_records)}')


Cross-model agreement computation (canonical samples) — 19:05:37


=== nsl_kdd_v2 / 5class_cw ===
  rf-xgb   K=10 agg: FA=0.721, RA=0.162, RC=0.569, PRA=0.720
  rf-dnn   K=10 agg: FA=0.456, RA=0.067, RC=0.015, PRA=0.515
  xgb-dnn   K=10 agg: FA=0.384, RA=0.043, RC=-0.193, PRA=0.433

=== nsl_kdd_v2 / 5class_smote ===
  rf-xgb   K=10 agg: FA=0.743, RA=0.153, RC=0.607, PRA=0.732
  rf-dnn   K=10 agg: FA=0.459, RA=0.053, RC=0.012, PRA=0.506
  xgb-dnn   K=10 agg: FA=0.389, RA=0.059, RC=-0.068, PRA=0.478

=== unsw_nb15_v2 / 5class_cw ===
  rf-xgb   K=10 agg: FA=0.699, RA=0.119, RC=0.536, PRA=0.707
  rf-dnn   K=10 agg: FA=0.470, RA=0.065, RC=0.189, PRA=0.570
  xgb-dnn   K=10 agg: FA=0.451, RA=0.084, RC=0.274, PRA=0.606

=== unsw_nb15_v2 / 5class_smote ===
  rf-xgb   K=10 agg: FA=0.701, RA=0.136, RC=0.576, PRA=0.725
  rf-dnn   K=10 agg: FA=0.482, RA=0.067, RC=0.169, PRA=0.565
  xgb-dnn   K=10 agg: FA=0.493, RA=0.077, RC=0.247, PRA=0.594

=== cic_ids2017_v2 / 5class_cw ===
  rf-xgb   K=10 agg: 

## 6. Build DataFrames and save

In [7]:
df_agg = pd.DataFrame(agg_results)
df_perclass = pd.DataFrame(perclass_results)
df_samples = pd.DataFrame(sample_disagreement_records)

out_dir = Path(REPO) / 'results' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

df_combined = pd.concat([df_agg, df_perclass], ignore_index=True, sort=False)
df_combined.to_csv(out_dir / 'krishna_agreement_canonical.csv', index=False)
print(f'✓ Saved: krishna_agreement_canonical.csv ({len(df_combined)} rows)')

df_samples.to_csv(out_dir / 'sample_disagreement_canonical.csv', index=False)
print(f'✓ Saved: sample_disagreement_canonical.csv ({len(df_samples)} rows)')

✓ Saved: krishna_agreement_canonical.csv (324 rows)
✓ Saved: sample_disagreement_canonical.csv (18000 rows)


## 7. Test v7 §12.7 predictions on canonical-aligned data

In [8]:
# PREDICTION 1: RF-XGB > RF-DNN and > XGB-DNN at K=10 aggregate (using RC as canonical agreement metric)
print('=' * 70)
print('PREDICTION 1: Tree-vs-tree agreement > tree-vs-DNN agreement (K=10 aggregate, RC)')
print('=' * 70)
k10_agg = df_agg[df_agg['k'] == 10]

pred1_results = []
for ds in DATASETS:
    for variant in MODEL_VARIANTS:
        sub = k10_agg[(k10_agg['dataset'] == ds) & (k10_agg['variant'] == variant)]
        if len(sub) < 3:
            continue
        rf_xgb = sub[sub['pair'] == 'rf-xgb'].iloc[0]
        rf_dnn = sub[sub['pair'] == 'rf-dnn'].iloc[0]
        xgb_dnn = sub[sub['pair'] == 'xgb-dnn'].iloc[0]

        rxa = rf_xgb['RC']
        rda = rf_dnn['RC']
        xda = xgb_dnn['RC']

        supports = rxa > max(rda, xda)
        pred1_results.append({
            'dataset': ds, 'variant': variant,
            'rf-xgb RC': round(rxa, 3),
            'rf-dnn RC': round(rda, 3),
            'xgb-dnn RC': round(xda, 3),
            'supports_pred1': supports,
        })
        print(f'  {ds:<18} {variant:<14} rf-xgb={rxa:.3f}, rf-dnn={rda:.3f}, xgb-dnn={xda:.3f}, supports={supports}')

supports_count = sum(1 for r in pred1_results if r['supports_pred1'])
print(f'\nPrediction 1 supported in {supports_count} of {len(pred1_results)} dataset-variant combos')

# PREDICTION 2: Agreement drops on rare classes (R2L, U2R) compared to common (Normal, DoS)
print('\n' + '=' * 70)
print('PREDICTION 2: Per-class RC drops on rare classes (R2L, U2R)')
print('=' * 70)
k10_pc = df_perclass[df_perclass['k'] == 10]

pred2_results = []
for ds in DATASETS:
    for variant in MODEL_VARIANTS:
        sub = k10_pc[(k10_pc['dataset'] == ds) & (k10_pc['variant'] == variant)]
        if len(sub) == 0:
            continue

        class_mean_rc = sub.groupby('class')['RC'].mean().to_dict()
        common = np.mean([class_mean_rc.get('Normal', np.nan), class_mean_rc.get('DoS', np.nan)])
        rare = np.mean([class_mean_rc.get(c, np.nan) for c in ['R2L', 'U2R'] if c in class_mean_rc])

        if np.isnan(rare) or np.isnan(common):
            supports = None
        else:
            supports = rare < common

        pred2_results.append({
            'dataset': ds, 'variant': variant,
            'common_class_RC (Normal/DoS)': round(common, 3) if not np.isnan(common) else None,
            'rare_class_RC (R2L/U2R)': round(rare, 3) if not np.isnan(rare) else None,
            'supports_pred2': supports,
        })
        print(f'  {ds:<18} {variant:<14} common_RC={common:.3f}, rare_RC={rare:.3f}, supports={supports}')

p2_yes = sum(1 for r in pred2_results if r['supports_pred2'] is True)
p2_total = sum(1 for r in pred2_results if r['supports_pred2'] is not None)
print(f'\nPrediction 2 supported in {p2_yes} of {p2_total} dataset-variant combos')

# Overall agreement stats
print('\n' + '=' * 70)
print('OVERALL AGREEMENT STATISTICS (K=10, aggregate, across all 18 model-pair combos)')
print('=' * 70)
for metric in ['FA', 'RA', 'SA', 'SRA', 'RC', 'PRA']:
    print(f'  {metric}: mean={k10_agg[metric].mean():.3f}, '
          f'min={k10_agg[metric].min():.3f}, '
          f'max={k10_agg[metric].max():.3f}')

PREDICTION 1: Tree-vs-tree agreement > tree-vs-DNN agreement (K=10 aggregate, RC)
  nsl_kdd_v2         5class_cw      rf-xgb=0.569, rf-dnn=0.015, xgb-dnn=-0.193, supports=True
  nsl_kdd_v2         5class_smote   rf-xgb=0.607, rf-dnn=0.012, xgb-dnn=-0.068, supports=True
  unsw_nb15_v2       5class_cw      rf-xgb=0.536, rf-dnn=0.189, xgb-dnn=0.274, supports=True
  unsw_nb15_v2       5class_smote   rf-xgb=0.576, rf-dnn=0.169, xgb-dnn=0.247, supports=True
  cic_ids2017_v2     5class_cw      rf-xgb=0.268, rf-dnn=-0.040, xgb-dnn=0.277, supports=False
  cic_ids2017_v2     5class_smote   rf-xgb=0.363, rf-dnn=0.001, xgb-dnn=0.303, supports=True

Prediction 1 supported in 5 of 6 dataset-variant combos

PREDICTION 2: Per-class RC drops on rare classes (R2L, U2R)
  nsl_kdd_v2         5class_cw      common_RC=0.342, rare_RC=0.293, supports=True
  nsl_kdd_v2         5class_smote   common_RC=0.329, rare_RC=0.293, supports=True
  unsw_nb15_v2       5class_cw      common_RC=0.316, rare_RC=0.255, suppor

## 8. Save summary JSON

In [10]:
summary = {
    'timestamp': datetime.now().isoformat(),
    'method': 'canonical_shared_samples_per_dataset',
    'n_datasets': len(DATASETS),
    'n_variants': len(MODEL_VARIANTS),
    'n_pairs': len(PAIRS),
    'n_k_values': len(K_VALUES),
    'n_classes': 5,
    'total_aggregate_rows': len(df_agg),
    'total_perclass_rows': len(df_perclass),
    'total_sample_records': len(df_samples),
    'k10_aggregate_stats': {
        m: {
            'mean': float(k10_agg[m].mean()),
            'min': float(k10_agg[m].min()),
            'max': float(k10_agg[m].max()),
            'median': float(k10_agg[m].median()),
        } for m in ['FA', 'RA', 'SA', 'SRA', 'RC', 'PRA']
    },
    'prediction_1_tree_vs_tree_higher': {
        'supported_count': sum(1 for r in pred1_results if r['supports_pred1']),
        'total_count': len(pred1_results),
        'details': pred1_results,
    },
    'prediction_2_rare_class_lower': {
        'supported_count': sum(1 for r in pred2_results if r['supports_pred2'] is True),
        'total_count': sum(1 for r in pred2_results if r['supports_pred2'] is not None),
        'details': pred2_results,
    },
}

out_path = Path(REPO) / 'results' / 'tables' / 'krishna_agreement_canonical_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✓ Saved: krishna_agreement_canonical_summary.json')

TypeError: Object of type bool is not JSON serializable

## 9. Display top per-class disagreement samples (Day 5 LLM evaluation input)

In [ ]:
print('=' * 70)
print('TOP-10 MOST-DISAGREED SAMPLES PER DATASET (across all pairs, K=10 aggregate)')
print('=' * 70)

for ds in DATASETS:
    sub = df_samples[df_samples['dataset'] == ds]
    if len(sub) == 0:
        continue
    per_sample_avg = sub.groupby('sample_position').agg({
        'disagreement_score': 'mean',
        'true_class': 'first',
    }).reset_index().sort_values('disagreement_score', ascending=False).head(10)

    print(f'\n{ds}:')
    for _, row in per_sample_avg.iterrows():
        cn = CLASS_NAMES_5[int(row['true_class'])]
        print(f'  sample_pos={int(row["sample_position"]):4d}, class={cn:<6}, mean disagreement={row["disagreement_score"]:.3f}')

## 10. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/06_krishna_agreement_v3.ipynb
!git add results/tables/krishna_agreement_canonical.csv
!git add results/tables/sample_disagreement_canonical.csv
!git add results/tables/krishna_agreement_canonical_summary.json
!git status --short
!git commit -m 'Notebook 06 v3: Krishna agreement metrics on canonical shared samples (18 models, 3 pairs, K={5,10,15}, aggregate+per-class+per-sample)'
!git push origin main